# Visualize enzyme protein graphs with IMSTE

Vectorize the active protein graph dataset in batches to limit memory use, then display an interactive 2D IMSTE layout. Set `GRAPH_BATCH_SIZE` below to tune peak memory use. EC labels color the plot after the unsupervised embedding fit; hover points for protein metadata.


In [1]:
from pathlib import Path
from importlib import reload
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks" / "src" / "enzyme_workflow").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root from this notebook's working directory")
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
from enzyme_workflow.common import add_project_sources
PROJECT_ROOT = add_project_sources(PROJECT_ROOT, include_imste=True)
import simplex_pair_vectorizer.vectorizer as vectorizer_module
reload(vectorizer_module)
import simplex_pair_vectorizer
reload(simplex_pair_vectorizer)
import enzyme_workflow.visualization as visualization
reload(visualization)
visualize_proteins_2d = visualization.visualize_proteins_2d

In [2]:
# User parameters
NODE_LABEL_ALPHABET = "aa20"
SIMPLEX_DIMENSION = (1,1)  # inclusive: integer means 0..max; tuple means (min, max)
CLIQUE_ENUMERATOR = "auto"  # auto: fast for dimensions 0–3; generic above 3; or force fast/generic
GRAPH_BATCH_SIZE = 128  # lower this to reduce peak memory use
PAIR_DISTANCE = (0,1)  # inclusive: integer means 0..max; tuple means (min, max)
NBITS = 11
INCLUDE_SELF = True
IMSTE_PARAMETERS = {
    "n_components": 2,
    "n_msts": 128,
    "n_epochs": 200,
    "gravity_strength": 0.1,
    "mean_shift_iterations": 0,
    "device": "auto",
    "random_state": 42,
    "pca_init": False,  # avoid sparse PCA SVD convergence failures on large feature matrices
    "verbose": True,
}
REBUILD_FEATURES = False
REBUILD_EMBEDDING = False


In [ ]:
embedding_result = visualize_proteins_2d(
    PROJECT_ROOT,
    node_label_alphabet=NODE_LABEL_ALPHABET,
    simplex_dimension=SIMPLEX_DIMENSION,
    clique_enumerator=CLIQUE_ENUMERATOR,
    graph_batch_size=GRAPH_BATCH_SIZE,
    pair_distance=PAIR_DISTANCE,
    nbits=NBITS,
    include_self=INCLUDE_SELF,
    imste_parameters=IMSTE_PARAMETERS,
    rebuild_features=REBUILD_FEATURES,
    rebuild_embedding=REBUILD_EMBEDDING,
)
display(embedding_result.embedding.head())
display(embedding_result.figure)


Loading graphs for vectorization (20,433 proteins)


Loading protein graphs:   0%|          | 0/20433 [00:00<?, ?protein/s]

Starting simplex-pair feature extraction


Vectorizing protein graphs:   0%|          | 0/20433 [00:00<?, ?protein/s]

IMSTE is transductive: the cached layout belongs to the exact graph set and row order. Feature matrices and coordinates are cached under `data/processed/embedding_cache/`; the interactive HTML plot is saved under `results/`.
